In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv() 

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)

from __future__ import annotations

import argparse
import math
import os
import random
import tempfile
from typing import Callable, Dict, List, Sequence

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from rf_common import (ENV_FIELDS, Config, FrozenScheduler, PdwSource, RunningStats, add_data_args,
                       composite_score, load_frozen_scheduler, make_env, probe_source, source_from_args)

class RandomScanner:
    """Baseline: uniformly picks K distinct bins at every step."""

    def __init__(self, num_bins, k_observed, seed=0):
        self.num_bins, self.k_observed = num_bins, k_observed
        self._rng = random.Random(seed)

    def reset(self):
        pass                                   

    def select_bins(self):
        return sorted(self._rng.sample(range(self.num_bins), self.k_observed))

    def observe(self, monitored_bins, obs):
        pass


class SequentialScanner:
    """Baseline: K consecutive bins per step, round-robin with wrap-around."""

    def __init__(self, num_bins, k_observed):
        self.num_bins, self.k_observed = num_bins, k_observed
        self.current_index = 0

    def reset(self):
        self.current_index = 0

    def select_bins(self):
        bins = [(self.current_index + i) % self.num_bins for i in range(self.k_observed)]
        self.current_index = (self.current_index + self.k_observed) % self.num_bins
        return bins

    def observe(self, monitored_bins, obs):
        pass


def run_policy(policy, cfg: Config, env, windows: List, snr: float, seed: int, debug: bool = True) -> Dict[str, float]:
    env.set_snr(snr)
    agg = RunningStats()
    for i, frames in enumerate(windows):
        env.reset(frames=frames, noise_seed=seed + 1000 * i)
        policy.reset()
        
        if debug and i == 0:
            print(f"\n--- [Debug] Tracking Bin Selections for Window {i} ---")
            
        for step in range(cfg.episode_length):
            bins = policy.select_bins()
            
            # Print the first 15 steps of the first episode
            if debug and i == 0 and step < 15:
                print(f"  Step {step:02d} | Selected Bins: {bins}")
                
            obs, reward, done, info, _, _, _ = env.step(bins)
            policy.observe(bins, obs)
            agg.add(info, cfg.k_observed, reward)
            if done:
                break
    return agg.summary()

def evaluate_all(policies: Dict[str, Callable[[], object]], cfg: Config, source: PdwSource, snr_list: Sequence[float],
                 n_windows: int, seed: int = 12345) -> pd.DataFrame:
    env = make_env(cfg, source, seed=seed)                    # eval env: streams from the EVAL source
    print(f"\n=== Streaming {n_windows} held-out windows once (shared by every policy and SNR) ===")
    windows = [env.fetch_episode() for _ in range(n_windows)]
    n_pulses = sum(len(fr) for w in windows for fr in w)
    print(f"  {n_pulses} in-band pulses across {n_windows} windows | {source.stats()}")
    rows = []
    for name, make in policies.items():
        for snr in snr_list:
            m = run_policy(make(), cfg, env, windows, snr, seed)   # fresh policy object per (policy, SNR)
            rows.append({"policy": name, "snr_db": snr, **m, "score": composite_score(m)})
    return pd.DataFrame(rows)


def print_tables(df: pd.DataFrame, nominal: float, snr_list: Sequence[float]) -> None:
    nom = df[df.snr_db == nominal].set_index("policy")
    print(f"\n=== Detection performance & scan efficiency @ {nominal:g} dB (same windows, same noise) ===")
    print(f"  {'policy':<34}{'Pd':>8}{'Pfa':>8}{'scan eff':>10}{'productive':>12}{'reward':>10}{'score':>8}")
    for name, r in nom.iterrows():
        print(f"  {name:<34}{r.pd:>8.3f}{r.pfa:>8.3f}{r.scan_eff:>10.3f}{r.productive_frac:>12.3f}"
              f"{r.base_reward:>10.1f}{r.score:>8.3f}")
    print("\n=== SNR sweep ===")
    print(f"  {'policy [metric]':<44}" + "".join(f"{s:>+9g}dB" for s in snr_list))
    for name in df.policy.unique():
        for key in ("pd", "pfa", "scan_eff"):
            vals = df[df.policy == name].sort_values("snr_db")[key]
            print(f"  {name + ' [' + key + ']':<44}" + "".join(f"{v:>11.3f}" for v in vals))


def plot_sweep(df: pd.DataFrame, path: str) -> None:
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
    for j, (key, ttl) in enumerate((("pd", "Pd"), ("pfa", "Pfa"), ("scan_eff", "Scan efficiency"))):
        for name in df.policy.unique():
            d = df[df.policy == name].sort_values("snr_db")
            style = "--x" if name in ("Random scanner", "Sequential scanner") else "-o"
            ax[j].plot(d.snr_db, d[key], style, label=name)
        ax[j].set(title=f"{ttl} vs SNR", xlabel="SNR (dB)")
    ax[0].legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"  saved plot: {path}")


def evaluate_checkpoints(ckpts: List[str], source: PdwSource, snrs: Sequence[float], n_windows: int,
                         out_dir: str, seed: int = 12345) -> pd.DataFrame:
    scheds: Dict[str, FrozenScheduler] = {}
    for p in ckpts:
        label = f"DQN [{os.path.basename(os.path.dirname(os.path.abspath(p))) or 'ckpt'}]"
        while label in scheds:
            label += "'"
        scheds[label] = load_frozen_scheduler(p)
    first = next(iter(scheds.values()))
    cfg = first.cfg
    for lab, s in scheds.items():                              # all checkpoints must share the env
        for f in ENV_FIELDS:
            if getattr(s.cfg, f) != getattr(cfg, f):
                raise ValueError(f"checkpoint '{lab}' differs from the first one in '{f}'; "
                                 f"they cannot be compared on the same windows")
    snr_list = sorted(set(list(snrs) + [cfg.snr_db]))

    policies: Dict[str, Callable[[], object]] = {name: (lambda s=s: s) for name, s in scheds.items()}
    policies["Random scanner"] = lambda: RandomScanner(cfg.num_bins, cfg.k_observed, seed=seed)
    policies["Sequential scanner"] = lambda: SequentialScanner(cfg.num_bins, cfg.k_observed)

    df = evaluate_all(policies, cfg, source, snr_list, n_windows, seed)
    print_tables(df, cfg.snr_db, snr_list)
    os.makedirs(out_dir, exist_ok=True)
    csv = os.path.join(out_dir, "eval_results.csv")
    df.to_csv(csv, index=False)
    print(f"  saved table: {csv}")
    plot_sweep(df, os.path.join(out_dir, "eval_snr_sweep.png"))
    return df


#Offline test

def selftest(out_dir: str) -> None:
    import fsspec
    from rf_common import H5StreamSource, write_mock_tsrd
    from train_dqn import save_frozen, train
    print("=== SELF-TEST (no network): mock TSRD -> train -> freeze -> evaluate vs baselines ===")
    cfg = Config(episode_length=60, frame_window_us=50.0, warmup_steps=120, eps_decay_steps=400,
                 buffer_capacity=2000, target_update_freq=50, use_ekf=True)
    with tempfile.TemporaryDirectory() as tmp:
        write_mock_tsrd(tmp, "stare", "train", 3)
        write_mock_tsrd(tmp, "stare", "validation", 2, seed=1)
        fs = fsspec.filesystem("file")
        tr = H5StreamSource(fs, tmp, "stare", "train", initial_rows=3000, verbose=False)
        va = H5StreamSource(fs, tmp, "stare", "validation", initial_rows=3000, verbose=False)
        try:
            probe_source(tr, cfg, 2)
            res = train(cfg, tr, "recurrent_ddqn", seed=0, num_episodes=6)
            ckpt = save_frozen(res, cfg, out_dir, "stare", "train", "local")
            evaluate_checkpoints([ckpt], va, [5, 10, 15], n_windows=3, out_dir=out_dir)
        finally:
            tr.close()
            va.close()
    print("SELF-TEST PASSED")


def main():
    ap = argparse.ArgumentParser(description="Evaluate frozen DQN schedulers against Random/Sequential scanners")
    add_data_args(ap, default_subset="validation", default_mode=None)
    ap.add_argument("--checkpoint", nargs="+", help="one or more dqn_scheduler_frozen.pt files")
    ap.add_argument("--eval-episodes", type=int, default=20)
    ap.add_argument("--snrs", type=float, nargs="+", default=[0, 5, 10, 15, 20])
    ap.add_argument("--seed", type=int, default=12345)
    ap.add_argument("--out-dir", default="eval_out")
    ap.add_argument("--selftest", action="store_true")
    a, _ = ap.parse_known_args(args=[
    "--checkpoint", r"C:\Rajiv Data Folder\Data\LLM Python\train_out\dqn_scheduler_frozen.pt",
    "--source", "hf",
    "--mode", "stare",
    "--subset", "val"
])
    if a.selftest:
        selftest(a.out_dir)
        return
    if not a.checkpoint:
        ap.error("--checkpoint is required (or use --selftest)")

    meta = load_frozen_scheduler(a.checkpoint[0]).meta
    train_mode, train_subset = meta.get("data_mode", "stare"), meta.get("train_subset", "train")
    mode = a.mode or train_mode
    if mode != train_mode:
        print(f"  WARNING: evaluating on '{mode}' but the checkpoint was trained on '{train_mode}'.")
    if a.subset == train_subset and mode == train_mode:
        print(f"  WARNING: evaluating on the same subset ('{a.subset}') the model was trained on - "
              f"use --subset validation or test for an honest comparison.")
    cfg = Config(**{k: (tuple(v) if k == "freq_range_ghz" else v) for k, v in meta.items()
                    if k in Config.__dataclass_fields__})
    print(f"  evaluating on TSRD {mode}/{a.subset}_{mode}  (trained on {train_mode}/{train_subset})")

    source = source_from_args(a, mode, a.subset)
    try:
        probe_source(source, cfg, 2)
        evaluate_checkpoints(a.checkpoint, source, a.snrs, a.eval_episodes, a.out_dir, a.seed)
    finally:
        source.close()


if __name__ == "__main__":
    main()